In [1]:
import pyemu
import os
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning) 
import pandas as pd
import matplotlib.pyplot as plt
import psutil
import shutil
import numpy as np
import sys
import swatmf
import matplotlib.pyplot as plt

In [2]:
swatmf.__version__

'1.0.1'

# 01. Set working directory

In [3]:
# path to project directory
prj_dir = "C:\\Users\\seonggpa\\Documents\\projects\\watersheds\\hbasnet_opt"
main_opt_path = os.path.join(prj_dir, 'main_opt')
os.chdir(main_opt_path)

# 02. Create prior pst

In [4]:
# create prior pst based on reweigted control file
pst = pyemu.Pst('mb_pp_rw.pst')

In [13]:
pst_prior = 'mb_pp_rw_prior.pst'

In [ ]:
# set prior UA
pst.pestpp_options['ies_drop_conflicts'] = True
pst.pestpp_options['ies_no_noise'] = True
pst.pestpp_options['ies_num_reals'] = 300 # number of realization
pst.control_data.noptmax = -1 # number of iteration
pst.model_command = 'python forward_run.py'
pst.write(f'{pst_prior}', version=2) # write new IES control file

noptmax:-1, npar_adj:83, nnz_obs:5114


In [8]:
os.chdir(os.pardir)

In [9]:
os.getcwd()

'C:\\Users\\seonggpa\\Documents\\projects\\watersheds\\hbasnet_opt'

# 04. Run IES

## 04.1 Set up IES

In [10]:
# check number of cores on your computer
num_workers = psutil.cpu_count(logical=False)

In [12]:
# name for prior ua directory
m_d = os.path.join(os.getcwd(), "mb_pp_rw_prior")

## 04.2 Execute

In [ ]:
pyemu.os_utils.start_workers(main_opt_path, # the folder which contains the "template" PEST dataset
                            'pestpp-ies', #the PEST software version we want to run
                            f'{pst_prior}', # the control file to use with PEST
                            num_workers=num_workers, #how many agents to deploy
                            worker_root='.', #where to deploy the agent directories; relative to where python is running
                            master_dir=m_d, #the manager directory,
                            # reuse_master=True
                            )

## 02-01 Change weights to make all of observation data visible

In [10]:
# reweight
pst = pyemu.Pst(pst_name)
pst.phi

np.float64(8052791.370865037)

In [11]:
# you can assign any values to balanced phi value for each group
balanced_groups = {grp:10000 for grp in pst.nnz_obs_groups}
pst.adjust_weights(obsgrp_dict=balanced_groups)

In [12]:
# Let's create a new control file with the number of iterations set to 30 and incorporate reweighted factors.
pst.control_data.noptmax = 30
pst.write(os.path.join(main_opt_path,'mb_pp_rw.pst'), version=2)

noptmax:30, npar_adj:83, nnz_obs:5114


In [13]:
os.getcwd()

'C:\\Users\\seonggpa\\Documents\\projects\\watersheds\\hbasnet_opt\\main_opt'

## TEST

In [14]:
pst_name = "mb_pp_rw.pst"

In [ ]:
#initial run
pyemu.os_utils.run(f'pestpp-glm.exe {pst_name}' , cwd=".")

pestpp-glm.exe mb_pp_rw.pst
